# Manipula FakeRecogna

In [2]:
%pip install openpyxl

/home/rafael/Projetos/campus_multiplataforma_llm/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import openpyxl

In [1]:
import pandas as pd
import config.prompts as PROMPTS
dataset = pd.read_excel("./FakeRecogna.xlsx")

MAX_TOKENS = 16384

"""PROMPT_TEMPLATE = {
                "role": "system",
                "content": PROMPTS.BASE["GKP"] # + ("\n\n" + PROMPTS.DEFINITION)
            }"""


'PROMPT_TEMPLATE = {\n                "role": "system",\n                "content": PROMPTS.BASE["GKP"] # + ("\n\n" + PROMPTS.DEFINITION)\n            }'

In [2]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 11903 entries, 0 to 11902
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Titulo     11872 non-null  str    
 1   Subtitulo  5580 non-null   str    
 2   Noticia    11902 non-null  str    
 3   Categoria  11902 non-null  str    
 4   Data       11551 non-null  object 
 5   Autor      11886 non-null  str    
 6   URL        11902 non-null  str    
 7   Classe     11902 non-null  float64
dtypes: float64(1), object(1), str(6)
memory usage: 11.3+ MB


In [3]:
dataset["Classe"].dropna().unique()

array([0., 1.])

In [4]:
dataset["Categoria"].unique()


<ArrowStringArray>
['entretenimento', 'saúde', 'mundo', 'ciência', 'brasil', 'política', nan]
Length: 7, dtype: str

In [5]:
dataset["Categoria"].value_counts(normalize=True).mul(100).round(2)

Categoria
saúde             37.44
política          33.20
entretenimento    11.84
brasil             7.60
ciência            5.06
mundo              4.87
Name: proportion, dtype: float64

In [6]:
dataset["Classe"].value_counts(normalize=True).mul(100).round(2)

Classe
0.0    50.0
1.0    50.0
Name: proportion, dtype: float64

In [7]:
# motando dataframe de amostra com 10% dos dados e proporção de 10/1 em classes 1
df_false = dataset[dataset["Classe"] == 0]
df_true = dataset[dataset["Classe"] == 1]

n_0 = len(df_true) // 100

amostra_1 = df_false.groupby("Categoria", group_keys=False).sample(n=n_0, random_state=42)

novo_df = pd.concat([df_true, amostra_1], axis=0)

In [8]:
novo_df["Classe"].value_counts(normalize=True).mul(100).round(2)

Classe
1.0    94.39
0.0     5.61
Name: proportion, dtype: float64

In [9]:
amostra = (
    novo_df.groupby(["Categoria", "Classe"], group_keys=False)
        .sample(frac=0.01, random_state=42)
)

In [10]:
amostra["Categoria"].value_counts(normalize=False)

Categoria
saúde             34
política          22
ciência            5
brasil             2
entretenimento     1
mundo              1
Name: count, dtype: int64

In [11]:
amostra["Classe"].value_counts(normalize=True).mul(100).round(2)

Classe
1.0    90.77
0.0     9.23
Name: proportion, dtype: float64

In [12]:
amostra.info()

<class 'pandas.DataFrame'>
Index: 65 entries, 8456 to 11901
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Titulo     65 non-null     str    
 1   Subtitulo  19 non-null     str    
 2   Noticia    65 non-null     str    
 3   Categoria  65 non-null     str    
 4   Data       63 non-null     object 
 5   Autor      64 non-null     str    
 6   URL        65 non-null     str    
 7   Classe     65 non-null     float64
dtypes: float64(1), object(1), str(6)
memory usage: 64.4+ KB


In [13]:
amostra.to_csv(f"./amostra_FakeRecogna_anomaly.csv", index=False)

# Desconsidera daqui pra baixo

In [41]:
def add_template (text: str, prompt: list[dict] = [PROMPT_TEMPLATE]) -> list[dict]:
    return prompt + [{"role": "user", "content": text}]

def token_counter(prompt: list[dict]) -> int:
    """Conta a quantidade de tokens aproximada um texto possui"""
    text = " ".join([msg["content"] for msg in prompt])
    estimated_tokens = max(1, len(text) // 4) + 879

    return estimated_tokens

prompt = [PROMPT_TEMPLATE]
i = 0
coluna = "Noticia"

while True:
    text = dataset.loc[i, coluna]
    prompt = add_template(text, prompt)

    if token_counter(prompt) > MAX_TOKENS:
        break

    i += 1
    
i

print(i)

NameError: name 'PROMPT_TEMPLATE' is not defined

In [ ]:
# Vamos dividir o dataset em chunks de tamanho i
list_df = [dataset[j:j+i] for j in range(0, len(dataset), i)]

print(f"Total de dataframes criados: {len(list_df)}")

In [ ]:
# Por fim, salvamos esses dataframes em arquivos CSV separados
for idx, df in enumerate(list_df):
    df.to_csv(f"./FakeRecogna_{idx+1}.csv")

print(f"Execução concluída. {len(list_df)} arquivos CSV foram criados com sucesso.")